# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos.copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
24,2022-09-02 00:00:00,0.000000,19,76,0,0,Noche,Noche,0.000000,0.000000
25,2022-09-02 01:00:00,0.000000,18,81,0,1,Noche,Noche,0.000000,0.000000
26,2022-09-02 02:00:00,0.000000,18,84,0,2,Noche,Noche,0.000000,0.000000
27,2022-09-02 03:00:00,0.000000,18,86,0,3,Noche,Noche,0.000000,0.000000
28,2022-09-02 04:00:00,0.000000,17,86,0,4,Noche,Noche,0.000000,0.000000
29,2022-09-02 05:00:00,0.000000,17,88,0,5,Noche,Noche,0.000000,0.000000
30,2022-09-02 06:00:00,0.000000,17,91,0,6,Soleado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,94,0,7,Soleado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,97,0,8,Soleado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,93,1,9,Soleado,Lluvioso,438.814997,7720.582326


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,19,76,0,0,0.0,0.0
25,18,81,0,1,0.0,0.0
26,18,84,0,2,0.0,0.0
27,18,86,0,3,0.0,0.0
28,17,86,0,4,0.0,0.0
...,...,...,...,...,...,...
18285,22,45,0,20,1450.0,0.0
18286,20,54,0,21,0.0,0.0
18287,18,62,0,22,0.0,0.0
18288,17,69,0,23,0.0,0.0


In [7]:
y = datos_dia[['Generación']]
y

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 12786, y_train: 12786
X_val: 2740, y_val: 2740
X_test: 2740, y_test: 2740


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.5        0.74736842 0.         0.         0.         0.        ]
 [0.47368421 0.8        0.         0.04347826 0.         0.        ]
 [0.47368421 0.83157895 0.         0.08695652 0.         0.        ]
 ...
 [0.36842105 0.77894737 0.07142857 0.60869565 0.71913333 0.34156667]
 [0.36842105 0.75789474 0.07142857 0.65217391 0.70556667 0.33793333]
 [0.36842105 0.72631579 0.07142857 0.69565217 0.79483333 0.24856667]]
(12786, 6)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.500000,0.747368,0.000000,0.000000,0.000000,0.000000
25,0.473684,0.800000,0.000000,0.043478,0.000000,0.000000
26,0.473684,0.831579,0.000000,0.086957,0.000000,0.000000
27,0.473684,0.852632,0.000000,0.130435,0.000000,0.000000
28,0.447368,0.852632,0.000000,0.173913,0.000000,0.000000
...,...,...,...,...,...,...
12805,0.342105,0.736842,0.142857,0.521739,0.843733,0.344767
12806,0.394737,0.652632,0.142857,0.565217,0.829000,0.338200
12807,0.368421,0.778947,0.071429,0.608696,0.719133,0.341567
12808,0.368421,0.757895,0.071429,0.652174,0.705567,0.337933


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.34210526 0.72631579 0.         0.73913043 0.70433333 0.24723333]
 [0.34210526 0.69473684 0.         0.7826087  0.702      0.2108    ]
 [0.34210526 0.69473684 0.         0.82608696 0.4138     0.01416667]
 ...
 [0.81578947 0.25263158 0.14285714 0.7826087  0.9109     0.82416667]
 [0.78947368 0.30526316 0.07142857 0.82608696 0.82416667 0.45596667]
 [0.73684211 0.35789474 0.         0.86956522 0.46683333 0.0394    ]]
(2740, 6)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12810,0.342105,0.726316,0.000000,0.739130,0.704333,0.247233
12811,0.342105,0.694737,0.000000,0.782609,0.702000,0.210800
12812,0.342105,0.694737,0.000000,0.826087,0.413800,0.014167
12813,0.315789,0.726316,0.000000,0.869565,0.043467,0.000000
12814,0.289474,0.768421,0.000000,0.913043,0.000000,0.000000
...,...,...,...,...,...,...
15545,0.894737,0.178947,0.357143,0.695652,0.959467,0.935400
15546,0.868421,0.200000,0.214286,0.739130,0.940767,0.910900
15547,0.815789,0.252632,0.142857,0.782609,0.910900,0.824167
15548,0.789474,0.305263,0.071429,0.826087,0.824167,0.455967


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.68421053 0.42105263 0.         0.91304348 0.0379     0.        ]
 [0.65789474 0.49473684 0.         0.95652174 0.         0.        ]
 [0.60526316 0.56842105 0.         1.         0.         0.        ]
 ...
 [0.47368421 0.6        0.         0.95652174 0.         0.        ]
 [0.44736842 0.67368421 0.         1.         0.         0.        ]
 [0.42105263 0.71578947 0.         0.         0.         0.        ]]
(2740, 6)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15550,0.684211,0.421053,0.0,0.913043,0.037900,0.0
15551,0.657895,0.494737,0.0,0.956522,0.000000,0.0
15552,0.605263,0.568421,0.0,1.000000,0.000000,0.0
15553,0.578947,0.600000,0.0,0.000000,0.000000,0.0
15554,0.578947,0.631579,0.0,0.043478,0.000000,0.0
...,...,...,...,...,...,...
18285,0.578947,0.421053,0.0,0.869565,0.048333,0.0
18286,0.526316,0.515789,0.0,0.913043,0.000000,0.0
18287,0.473684,0.600000,0.0,0.956522,0.000000,0.0
18288,0.447368,0.673684,0.0,1.000000,0.000000,0.0


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.48717949 0.75257732 0.         0.         0.         0.        ]
 [0.46153846 0.80412371 0.         0.04347826 0.         0.        ]
 [0.46153846 0.83505155 0.         0.08695652 0.         0.        ]
 ...
 [0.46153846 0.60824742 0.         0.95652174 0.         0.        ]
 [0.43589744 0.68041237 0.         1.         0.         0.        ]
 [0.41025641 0.72164948 0.         0.         0.         0.        ]]
(18266, 6)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.487179,0.752577,0.0,0.000000,0.000000,0.0
25,0.461538,0.804124,0.0,0.043478,0.000000,0.0
26,0.461538,0.835052,0.0,0.086957,0.000000,0.0
27,0.461538,0.855670,0.0,0.130435,0.000000,0.0
28,0.435897,0.855670,0.0,0.173913,0.000000,0.0
...,...,...,...,...,...,...
18285,0.564103,0.432990,0.0,0.869565,0.048333,0.0
18286,0.512821,0.525773,0.0,0.913043,0.000000,0.0
18287,0.461538,0.608247,0.0,0.956522,0.000000,0.0
18288,0.435897,0.680412,0.0,1.000000,0.000000,0.0


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.        ]
 ...
 [0.70556667]
 [0.79483333]
 [0.70433333]]
(12786, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
24,0.000000
25,0.000000
26,0.000000
27,0.000000
28,0.000000
...,...
12805,0.829000
12806,0.719133
12807,0.705567
12808,0.794833


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.702     ]
 [0.4138    ]
 [0.04346667]
 ...
 [0.82416667]
 [0.46683333]
 [0.0379    ]]
(2740, 1)


In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12810,0.702000
12811,0.413800
12812,0.043467
12813,0.000000
12814,0.000000
...,...
15545,0.940767
15546,0.910900
15547,0.824167
15548,0.466833


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(2740, 1)


In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15550,0.0
15551,0.0
15552,0.0
15553,0.0
15554,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(18266, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (12738, 48, 6), y_train: (12738, 1)
X_val: (2692, 48, 6), y_val: (2692, 1)
X_test: (2692, 48, 6), y_test: (2692, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.4 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 13:09:41,758] A new study created in memory with name: no-name-43a82683-baf7-46c0-93ce-36fa6045715a


[LightGBM] [Warning] min_data_in_leaf is set=55, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=55
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=55, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=55
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001229 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 684
[LightGBM] [Info] Number of data points in the train set: 12786, number of used features: 6
[LightGBM] [Info] Start training from score 0.296336
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

[I 2025-03-14 13:09:42,007] Trial 0 finished with value: 0.0038131394456136936 and parameters: {'num_leaves': 217, 'subsample': 0.942125905191218, 'colsample_bytree': 0.2636178572174231, 'min_data_in_leaf': 55}. Best is trial 0 with value: 0.0038131394456136936.


[LightGBM] [Warning] min_data_in_leaf is set=55, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=55
[LightGBM] [Warning] min_data_in_leaf is set=27, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=27
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=27, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=27
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 684
[LightGBM] [Info] Number of data points in the train set: 12786, number of used features: 6
[LightGBM] [Info] Start training from score 0.296336


[I 2025-03-14 13:09:42,388] Trial 1 finished with value: 0.004260791656573095 and parameters: {'num_leaves': 198, 'subsample': 0.14555103892916654, 'colsample_bytree': 0.6030348412393586, 'min_data_in_leaf': 27}. Best is trial 0 with value: 0.0038131394456136936.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 13:09:55,047] Trial 48 finished with value: 0.0033724366203037857 and parameters: {'num_leaves': 441, 'subsample': 0.3070504758894707, 'colsample_bytree': 0.45850476343131685, 'min_data_in_leaf': 76}. Best is trial 14 with value: 0.003363233152905849.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 13:09:55,211] Trial 49 finished with value: 0.0037358897068034596 and parameters: {'num_leaves': 431, 'subsample': 0.31052297093775166, 'colsample_bytree': 0.2992730933542961, 'min_data_in_leaf': 88}. Best is trial 14 with value: 0.003363233152905849.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 13:09:55,218] A new study created in memory with name: no-name-1ce79b9e-0050-4645-b5b4-f7e8a7f5d5b1
[I 2025-03-14 13:10:05,751] Trial 0 finished with value: 0.005385874838689301 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 0 with value: 0.005385874838689301.
[I 2025-03-14 13:10:14,915] Trial 1 finished with value: 0.004917310463418245 and parameters: {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 13, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 1 with value: 0.004917310463418245.
[I 2025-03-14 13:10:22,423] Trial 2 finished with value: 0.00402880044825455 and parameters: {'n_estimators': 300, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 2 with value: 0.00402880044825455.
[I 2025-03-14 13:10:26,607] Trial 3 finished with value: 0.004088667340438525 and parameters: {'n_estimators': 150, 'max_depth': 50, 'm

Mejores hiperparámetros: {'n_estimators': 500, 'max_depth': 10, 'min_samples_split': 17, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [38]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [39]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [40]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 13:15:34,169] A new study created in memory with name: no-name-5b2d6f29-ce2c-403a-bf33-5ca59478bf09
[I 2025-03-14 13:16:45,096] Trial 0 finished with value: 0.004758260678499937 and parameters: {'head_size': 4, 'num_heads': 5, 'ff_dim': 96, 'num_transformer_blocks': 3, 'mlp_units_1': 128, 'mlp_units_2': 256, 'dropout': 0.31752704914370833, 'mlp_dropout': 0.3202547626842649, 'learning_rate': 0.0005097217187145229, 'batch_size': 128}. Best is trial 0 with value: 0.004758260678499937.
[I 2025-03-14 13:18:07,085] Trial 1 finished with value: 0.006694264244288206 and parameters: {'head_size': 4, 'num_heads': 5, 'ff_dim': 96, 'num_transformer_blocks': 4, 'mlp_units_1': 128, 'mlp_units_2': 256, 'dropout': 0.4600449923020056, 'mlp_dropout': 0.3976423756830637, 'learning_rate': 8.853522368441598e-05, 'batch_size': 512}. Best is trial 0 with value: 0.004758260678499937.
[I 2025-03-14 13:18:56,807] Trial 2 finished with value: 0.004530934151262045 and parameters: {'head_size': 8, 'n

Mejores hiperparámetros: {'head_size': 8, 'num_heads': 6, 'ff_dim': 64, 'num_transformer_blocks': 1, 'mlp_units_1': 320, 'mlp_units_2': 224, 'dropout': 0.2185773791153159, 'mlp_dropout': 0.15181852511491634, 'learning_rate': 0.0023194796521910973, 'batch_size': 128}


### Forescasting

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 14:04:23,516] A new study created in memory with name: no-name-9549bd5b-d827-4a68-8107-5a57367092aa
[I 2025-03-14 14:05:48,137] Trial 10 finished with value: 0.28832679986953735 and parameters: {'filters': 32, 'kernel_size': 3, 'lstm_units_1': 128, 'lstm_units_2': 64, 'lstm_units_3': 64, 'dropout_lstm': 0.2903155584390058, 'dropout_dense': 0.3172984962047719, 'learning_rate': 0.00019337727610017308, 'batch_size': 128}. Best is trial 10 with value: 0.28832679986953735.
[I 2025-03-14 14:06:01,404] Trial 12 finished with value: 0.2674502432346344 and parameters: {'filters': 128, 'kernel_size': 3, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 16, 'dropout_lstm': 0.17191452600486362, 'dropout_dense': 0.42007679508449336, 'learning_rate': 0.0004462046513413169, 'batch_size': 256}. Best is trial 12 with value: 0.2674502432346344.
[I 2025-03-14 14:06:18,172] Trial 13 finished with value: 0.2899434566497803 and parameters: {'filters': 32, 'kernel_size': 2, 'lstm_units_1

Mejores hiperparámetros: {'filters': 32, 'kernel_size': 3, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 64, 'dropout_lstm': 0.2198136912202202, 'dropout_dense': 0.1055940653984536, 'learning_rate': 0.005884684263161474, 'batch_size': 128}


### Photovoltaic

In [42]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 14:31:38,545] A new study created in memory with name: no-name-12ce0898-7e10-494a-8cc2-6717549f10ce


Epoch 57: early stopping
Restoring model weights from the end of the best epoch: 47.


[I 2025-03-14 14:34:56,712] Trial 11 finished with value: 0.0054115657694637775 and parameters: {'filters_1': 128, 'filters_2': 64, 'kernel_size': 2, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.2147649143097429, 'dense_units_1': 128, 'dense_units_2': 16, 'learning_rate': 0.0002543356199225852, 'batch_size': 512}. Best is trial 11 with value: 0.0054115657694637775.


Epoch 60: early stopping
Restoring model weights from the end of the best epoch: 50.


[I 2025-03-14 14:35:05,643] Trial 9 finished with value: 0.0041771503165364265 and parameters: {'filters_1': 32, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.23894843113163136, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.001130176861737509, 'batch_size': 512}. Best is trial 9 with value: 0.0041771503165364265.


...

Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 22.


[I 2025-03-14 14:47:11,559] Trial 48 finished with value: 0.00454168114811182 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.20549831033245952, 'dense_units_1': 128, 'dense_units_2': 64, 'learning_rate': 0.0018994945848870956, 'batch_size': 512}. Best is trial 9 with value: 0.0041771503165364265.


Epoch 40: early stopping
Restoring model weights from the end of the best epoch: 30.


[I 2025-03-14 14:47:15,081] Trial 49 finished with value: 0.004561691544950008 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.4144272992117005, 'dense_units_1': 128, 'dense_units_2': 64, 'learning_rate': 0.004191729623241541, 'batch_size': 512}. Best is trial 9 with value: 0.0041771503165364265.


Mejores hiperparámetros: {'filters_1': 32, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.23894843113163136, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.001130176861737509, 'batch_size': 512}
